# setup + imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import json

from sklearn.model_selection import GroupKFold

In [2]:
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from catboost import CatBoostRegressor

In [3]:
from sklearn.metrics import r2_score
from scipy.stats import pearsonr

In [4]:
train_data = pd.read_csv('train_data.csv')
test_data = pd.read_csv('test_data.csv')
print(train_data.shape, test_data.shape)

(135210, 5) (9425, 5)


In [5]:
X = train_data.drop(columns='answer')
y = train_data['answer']

# data cleaning + feature engineering

In [6]:
def cleanword(w):
    w = str(w).lower().strip('.,;:!?()"„”')
    return w

In [7]:
def upgradedata(data):
    
    data = data.copy()
    genre = []
    page = []
    position = []
    for wid in data['word_id']:
        parts = wid.split('_')
        genre.append(parts[0])                 
        page.append(int(parts[-2]))            
        position.append(int(parts[-1]))
    data['genre'] = genre
    data['page'] = page
    data['position'] = position

    neword = []
    for w in data['word']:
        neword.append(cleanword(w))
    data.drop(columns='word')
    data['word'] = neword

    length = []
    for w in data['word']:
        length.append(len(w))
    data['length'] = length

    fullw = pd.concat([train_data[['word_id', 'word']], test_data[['word_id', 'word']]]).drop_duplicates('word_id')
    frecv = {}
    for w in fullw['word']:
        w = cleanword(w)
        if w not in frecv:
            frecv[w] = 1
        else:
            frecv[w] += 1

    fcol = []
    for w in data['word']:
        fcol.append(frecv[w])
    data['frecv'] = fcol
    
    return data

In [8]:
X = upgradedata(X)
test_data = upgradedata(test_data)

In [9]:
def addmean(X_fit, y_fit, X_other):
    X_fit = X_fit.copy()
    X_other = X_other.copy()
    s_all = y_fit.groupby(X_fit['word']).transform('sum')                      # this word, everywhere
    c_all = y_fit.groupby(X_fit['word']).transform('count')
    s_doc = y_fit.groupby([X_fit['word'], X_fit['text']]).transform('sum')     # this word, same document
    c_doc = y_fit.groupby([X_fit['word'], X_fit['text']]).transform('count')
    X_fit['mean'] = ((s_all - s_doc) / (c_all - c_doc)).replace([np.inf, -np.inf], np.nan)
    word_mean = y_fit.groupby(X_fit['word']).mean()
    X_other['mean'] = X_other['word'].map(word_mean)
    return X_fit, X_other

# data exploration

In [10]:
print(test_data['participant_id'].isin(train_data['participant_id']).mean())

0.0


In [11]:
print('test words also in train:', test_data['word'].isin(train_data['word']).mean())

test words also in train: 0.5273209549071618


In [12]:
print('test word_ids also in train:', test_data['word_id'].isin(train_data['word_id']).mean())

test word_ids also in train: 0.0


# validation

In [13]:
def metric(y, preds):
    y = y.astype(float)
    preds = preds.astype(float)
    r2 = r2_score(y, preds, sample_weight=None, force_finite=True)
    r2 = max(0, r2)
    pears = pearsonr(y, preds)[0]
    if np.isnan(pears):
        pears = 0.0
    pears = np.abs(pears)
    return 100*(pears + r2)/2


def evaluate(model, X, y):
    kf = GroupKFold(n_splits=5)
    groups = train_data['text']
    
    scores = []
    for fit_rows, val_rows in kf.split(X, y, groups):
        X_fit = X.iloc[fit_rows]
        y_fit = y.iloc[fit_rows]
        X_val = X.iloc[val_rows]
        y_val = y.iloc[val_rows]
        X_fit, X_val = addmean(X_fit, y_fit, X_val)

        model.fit(X_fit[catfeats], y_fit)                  
        preds = model.predict(X_val[catfeats])
        scores.append(metric(y_val, preds))
    
    print('folds:', [round(s, 3) for s in scores])
    print('mean:', round(np.mean(scores), 4), ' spread:', round(np.std(scores), 4))
    return scores
    

# model

In [14]:
catfeats = ['length', 'frecv', 'page', 'position', 'genre', 'mean']
scores2 = evaluate(CatBoostRegressor(cat_features=['genre'], iterations=300, verbose=0, random_seed=0), X, y)

folds: [np.float64(37.281), np.float64(33.814), np.float64(21.093), np.float64(29.56), np.float64(35.9)]
mean: 31.5296  spread: 5.8344


In [15]:
scores2 = evaluate(CatBoostRegressor(cat_features=['genre'], iterations=300, verbose=0, random_seed=0), X, y)
scores2

folds: [np.float64(37.281), np.float64(33.814), np.float64(21.093), np.float64(29.56), np.float64(35.9)]
mean: 31.5296  spread: 5.8344


[np.float64(37.2813909454573),
 np.float64(33.81378253775007),
 np.float64(21.093200170905007),
 np.float64(29.559589616059906),
 np.float64(35.90012804622005)]

In [16]:
model = CatBoostRegressor(cat_features=['genre'], iterations=300, verbose=0, random_seed=0)

X_all, test_feat = addmean(X, y, test_data)
model.fit(X_all[catfeats], y)
preds = model.predict(test_feat[catfeats])

# submission

In [17]:
answers = pd.DataFrame({
    'subtaskID': 1,
    'datapointID': test_data['datapointID'],
    'answer': preds
})     

answers.to_csv('output.csv', index=False)